In [0]:
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType, LongType, TimestampType

# 1. Create a master calendar of business days from 2000 to last available date
max_date = spark.read.table("db_miniproject_finance_ws.bronze.sp500_initial").agg(F.max("Date")).collect()[0][0]
date_range = pd.date_range(start="2000-01-01", end=max_date, freq="B")
master_df = pd.DataFrame({"Date": date_range})

# 2. Convert to a Spark DataFrame
master_spark_df = spark.createDataFrame(master_df)

# 3. Function for joining stock tables

def join_stock_to_master(table_name, master_df):
  """Reads a raw bronze stock table, ensures Date is a timestamp,

  and left-joins it onto the master calendar.
  """
  # 3.1. Read the raw stock table
  raw_df = spark.read.table(table_name)

  # 3.2. Normalize Date column to timestamp
  raw_df = raw_df.withColumn("Date", F.to_timestamp("Date"))

  # 3.3. Perform left join with the master calendar
  final_df = master_df.join(raw_df, on="Date", how="left")
  
  return final_df

# 4. Function for cleaning stock tables

def clean_stock_df(df):
  """Enforces explicit data types, casts timestamps,

  and adds the processed timestamp.
  """
  return (
      df.withColumn("Date", F.col("Date").cast(TimestampType()))
      .withColumn("Close", F.col("Close").cast(FloatType()))
      .withColumn("High", F.col("High").cast(FloatType()))
      .withColumn("Low", F.col("Low").cast(FloatType()))
      .withColumn("Open", F.col("Open").cast(FloatType()))
      .withColumn("Volume", F.col("Volume").cast(LongType()))
      .withColumn(
          "ingestion_timestamp",
          F.col("ingestion_timestamp").cast(TimestampType()),
      )
      .withColumn("processed_timestamp", F.current_timestamp())
  )

In [0]:
# 5. Merge bronze tables with master calendar

final_stock_usa_df = join_stock_to_master("bronze.sp500_initial", master_spark_df)
final_stock_eu_df = join_stock_to_master("bronze.stoxx_eu_initial", master_spark_df)
final_stock_chn_df = join_stock_to_master("bronze.ssec_initial", master_spark_df)
final_stock_jpn_df = join_stock_to_master("bronze.nikkei_initial", master_spark_df)
final_stock_ind_df = join_stock_to_master("bronze.nsei_initial", master_spark_df)

In [0]:
# 6. Duplicate date check

stock_dfs = {
    "USA (S&P 500)": final_stock_usa_df,
    "Europe (STOXX)": final_stock_eu_df,
    "China (SSEC)": final_stock_chn_df,
    "Japan (Nikkei)": final_stock_jpn_df,
    "India (NSEI)": final_stock_ind_df,
}

for name, df in stock_dfs.items():
    before_count = df.count()
    deduped_df = df.dropDuplicates(["Date"])
    after_count = deduped_df.count()
    dropped = before_count - after_count
    print(f"{name}: {dropped} duplicate date(s) dropped ({before_count} → {after_count} rows)")
    stock_dfs[name] = deduped_df

final_stock_usa_df = stock_dfs["USA (S&P 500)"]
final_stock_eu_df = stock_dfs["Europe (STOXX)"]
final_stock_chn_df = stock_dfs["China (SSEC)"]
final_stock_jpn_df = stock_dfs["Japan (Nikkei)"]
final_stock_ind_df = stock_dfs["India (NSEI)"]

print("\nDuplicate dates removed from all 5 stock dataframes!")

In [0]:
# 7. Clean stock tables

cleaned_stock_usa_df = clean_stock_df(final_stock_usa_df).orderBy("Date")
cleaned_stock_eu_df = clean_stock_df(final_stock_eu_df).orderBy("Date")
cleaned_stock_chn_df = clean_stock_df(final_stock_chn_df).orderBy("Date")
cleaned_stock_jpn_df = clean_stock_df(final_stock_jpn_df).orderBy("Date")
cleaned_stock_ind_df = clean_stock_df(final_stock_ind_df).orderBy("Date")

print("All 5 individual stock dataframes successfully cleaned and typed!")

In [0]:
# 8. Write to Silver

# Write USA (S&P 500) to Silver
cleaned_stock_usa_df.write.format("delta").mode("overwrite").saveAsTable(
    "silver.sp500_cleaned"
)

# Write Europe (STOXX) to Silver
cleaned_stock_eu_df.write.format("delta").mode("overwrite").saveAsTable(
    "silver.stoxx_eu_cleaned"
)

# Write China (SSEC) to Silver
cleaned_stock_chn_df.write.format("delta").mode("overwrite").saveAsTable(
    "silver.ssec_cleaned"
)

# Write Japan (Nikkei) to Silver
cleaned_stock_jpn_df.write.format("delta").mode("overwrite").saveAsTable(
    "silver.nikkei_cleaned"
)

# Write India (Nisei/Nifty) to Silver
cleaned_stock_ind_df.write.format("delta").mode("overwrite").saveAsTable(
    "silver.nsei_cleaned"
)

print("All 5 stock index tables successfully written to the Silver layer!")

In [0]:
# 9. ZORDER for performance optimization

tables = [
    "silver.sp500_cleaned",
    "silver.stoxx_eu_cleaned",
    "silver.ssec_cleaned",
    "silver.nikkei_cleaned",
    "silver.nsei_cleaned",
]

for table in tables:
  spark.sql(f"OPTIMIZE {table} ZORDER BY (Date)")
  print(f"Optimized {table}")